# 02 - Prétraitement des données

Objectif : nettoyer le texte des reviews et corriger les anomalies de données repérées au notebook 01 (EDA), avant l'extraction par regex (03) et la classification Naive Bayes (04).

In [1]:
import pandas as pd
import numpy as np
import re
import html
import spacy

pd.set_option("display.max_colwidth", 120)

In [2]:
df_train = pd.read_csv("../data/drugsComTrain_raw.csv")
df_test = pd.read_csv("../data/drugsComTest_raw.csv")

df_train["split"] = "train"
df_test["split"] = "test"
df = pd.concat([df_train, df_test], ignore_index=True)
df.shape

(215063, 8)

## 1. Correction de l'anomalie sur `condition`

Repérée au notebook 01 : certains libellés de condition sont tronqués dans le dataset source (ex. `Bipolar Disorde` au lieu de `Bipolar Disorder`, `ibromyalgia` au lieu de `Fibromyalgia`).

In [3]:
def fix_condition(cond):
    if pd.isna(cond):
        return cond
    cond = re.sub(r"Disorde\b", "Disorder", cond)
    if cond == "ibromyalgia":
        cond = "Fibromyalgia"
    return cond

df["condition"] = df["condition"].apply(fix_condition)

# Vérification : ces libellés ne devraient plus exister
remaining = df["condition"].dropna().unique()
suspects = [c for c in remaining if c.endswith("Disorde") or c == "ibromyalgia"]
print(f"Libellés encore tronqués : {len(suspects)}")

Libellés encore tronqués : 0


## 2. Nettoyage du texte des reviews

- Les reviews sont entourées de guillemets littéraux (`"..."`) provenant du scraping d'origine.
- Elles contiennent des entités HTML (`&#039;`) et des retours à la ligne `\r\n`.
- On applique un nettoyage de base avant la tokenisation.

In [4]:
def clean_text(text):
    text = str(text)
    text = html.unescape(text)                 # &#039; -> '
    text = text.strip()
    if text.startswith('"') and text.endswith('"'):
        text = text[1:-1]
    text = re.sub(r"\s+", " ", text)          # \r\n, espaces multiples -> un seul espace
    return text.strip()

df["review_clean"] = df["review"].apply(clean_text)
df[["review", "review_clean"]].head(3)

,review,review_clean
0,"""It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil""","It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"
1,"""My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he sta...","My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he star..."
2,"""I used to take another oral contraceptive, which had 21 pill cycle, and was very happy- very light periods, max 5 d...","I used to take another oral contraceptive, which had 21 pill cycle, and was very happy- very light periods, max 5 da..."


## 3. Tokenisation, lemmatisation et suppression des stopwords (spaCy)

Sur ~215k lignes, le pipeline complet de `en_core_web_sm` (tagger + parser + ner) est trop lent (~30 minutes). On n'a besoin que de la tokenisation et de la lemmatisation, donc on exclut les composants inutiles avec `exclude=` (pas juste `disable=`, qui les garde chargés en mémoire) et on utilise un lemmatiseur en mode `lookup` (dictionnaire, ne nécessite pas le tagger). Résultat : ~2 minutes pour tout le corpus.

In [5]:
nlp = spacy.load(
    "en_core_web_sm",
    exclude=["tok2vec", "tagger", "parser", "senter", "attribute_ruler", "lemmatizer", "ner"],
)
nlp.add_pipe("lemmatizer", config={"mode": "lookup"}).initialize()

def preprocess(doc):
    tokens = [
        tok.lemma_.lower()
        for tok in doc
        if not tok.is_stop and not tok.is_punct and not tok.is_space and tok.is_alpha
    ]
    return " ".join(tokens)

# nlp.pipe pour traiter le corpus par lots (plus rapide que .apply ligne par ligne)
df["review_tokens"] = [
    preprocess(doc)
    for doc in nlp.pipe(df["review_clean"].tolist(), batch_size=256, n_process=1)
]
df[["review_clean", "review_tokens"]].head(3)

,review_clean,review_tokens
0,"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil",effect combination bystolic mg fish oil
1,"My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he star...",son halfway 4 week intuniv concern begin week start take high dose day hardly bed cranky sleep nearly hour drive hom...
2,"I used to take another oral contraceptive, which had 21 pill cycle, and was very happy- very light periods, max 5 da...",oral contraceptive pill cycle light period max day effect contain hormone gestodene available switch lybrel ingredie...


## 4. Vérification rapide de l'impact du nettoyage

In [6]:
df["tokens_count"] = df["review_tokens"].str.split().str.len()
df[["review_len_words_avant", "tokens_count"]] = pd.DataFrame({
    "review_len_words_avant": df["review_clean"].str.split().str.len(),
    "tokens_count": df["tokens_count"],
})
df[["review_len_words_avant", "tokens_count"]].describe()

,review_len_words_avant,tokens_count
count,215063.000000,215063.000000
mean,84.630183,33.635353
std,44.866092,18.002070
min,1.000000,0.000000
25%,48.000000,19.000000
50%,84.000000,33.000000
75%,126.000000,49.000000
max,1894.000000,765.000000


In [7]:
# Exemple avant/après complet
sample = df.sample(1, random_state=42).iloc[0]
print("AVANT :", sample["review"][:300])
print()
print("APRÈS :", sample["review_tokens"][:300])

AVANT : "I suffered a .45 caliber GSW to the head in 2009. Have had multiple OMFS/ENT surgeries. I have developed a chronic pain condition as a result of the accident. This medication has been a lifesaver. It allows me to function at work and has restored my quality of life. For me, it works significantly b

APRÈS : suffer caliber gsw head multiple omfs ent surgery develope chronic pain condition result accident medication lifesaver allow function work restore quality life work significantly well percocet


## 5. Sauvegarde du dataset prétraité

Le fichier est écrit dans `data/` (non versionné, voir `.gitignore`) pour être réutilisé par les notebooks 03 et 04.

In [8]:
cols_to_keep = [
    "uniqueID", "split", "drugName", "condition", "rating", "date", "usefulCount",
    "review", "review_clean", "review_tokens",
]
df[cols_to_keep].to_csv("../data/drugs_preprocessed.csv", index=False)
print("Sauvegardé :", df.shape)

Sauvegardé : (215063, 12)


## Résumé / prochaines étapes

- Conditions tronquées corrigées.
- Reviews nettoyées (entités HTML, guillemets, espaces) puis tokenisées/lemmatisées/dé-stopwordisées avec spaCy.
- Dataset prétraité sauvegardé dans `data/drugs_preprocessed.csv`.
- Prochaine étape (`03_regex.ipynb`) : affiner la détection des cas "urgents" par expressions régulières sur `review_clean` (texte non lemmatisé, pour préserver les tournures comme *can't breathe*).